# PMW · UVR5 人声与伴奏分离
运行下方单元，粘贴 PMW 提供的配对码。保持 PMW 页面开启。分离后自动回传音轨，PMW 接续 RunPod 换声。

In [ ]:
import getpass
import re
import shutil
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import requests

RELAY = "https://pmw-colab-relay.wilkong4876.chatgpt.site"
code = getpass.getpass("粘贴 PMW 配对码：").strip()
if not re.fullmatch(r"[0-9a-f-]{36}\.[0-9a-f]{64}", code):
    raise ValueError("配对码格式不正确，请从 PMW 重新复制。")
task_id, token = code.split(".")
base = f"{RELAY}/transfer/{task_id}"
headers = {"X-PMW-Colab": token}

def check_response(response):
    if response.status_code == 403:
        raise RuntimeError("配对码已过期或任务已结束，请检查 PMW。")
    if response.status_code == 413:
        raise RuntimeError("音轨超过 PMW 的 95 MB 限制。")
    if not response.ok:
        raise RuntimeError(f"传输失败（HTTP {response.status_code}），可重新运行本单元。")

with TemporaryDirectory(prefix="pmw-uvr5-") as temp:
    folder = Path(temp)
    print("1/4 从 PMW 读取歌曲…", flush=True)
    original = folder / "source"
    with requests.get(base + "/source", headers=headers, stream=True, timeout=(15, 90)) as response:
        check_response(response)
        with original.open("wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                file.write(chunk)
    song = folder / "song.wav"
    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", "-i", str(original),
                    "-map", "0:a:0", "-acodec", "pcm_s16le", str(song)], check=True)

    print("2/4 准备 UVR5…", flush=True)
    gpu = shutil.which("nvidia-smi") and subprocess.run(
        ["nvidia-smi", "-L"], capture_output=True
    ).returncode == 0
    device = "gpu" if gpu else "cpu"
    if not gpu:
        print("本次使用 CPU，分离会较慢。", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"audio-separator[{device}]==0.30.2"], check=True)

    print("3/4 分离人声和伴奏…", flush=True)
    separator = """
import sys
from audio_separator.separator import Separator
song, out, model_dir = sys.argv[1:]
separator = Separator(output_dir=out, output_format='WAV', model_file_dir=model_dir)
separator.load_model(model_filename='UVR_MDXNET_KARA_2.onnx')
separator.separate(song, custom_output_names={'Vocals':'vocals','Instrumental':'instrumental'})
"""
    subprocess.run([sys.executable, "-c", separator, str(song), temp, "/content/pmw-uvr-models"], check=True)

    print("4/4 回传两条音轨，PMW 会自动接续换声…", flush=True)
    for role, name in (("vocal", "vocals.wav"), ("backing", "instrumental.wav")):
        with (folder / name).open("rb") as audio:
            with requests.put(base + "/" + role, headers={**headers, "Content-Type": "audio/wav"},
                              data=audio, timeout=(15, 180)) as response:
                check_response(response)
del code, token, headers
print("已回传。请返回 PMW 查看换声与 WAV 入库进度。")
